<a href="https://colab.research.google.com/github/rghdzarad/31109170100123_Library/blob/main/phone_addiction_prediction_competition_learning_purpose.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Phone addiction prediction (AI & Machine Learning)**

# **Preparing for the usage of the data**

importing all the needed libraries

In [5]:
print("Importing the needed libraries for the coding process..................")
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, classification_report
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder , LabelEncoder
from sklearn.impute import SimpleImputer
import time
time.sleep(0.5) # delay
print("The needed libraries were imported successfully :)")

Importing the needed libraries for the coding process..................
The needed libraries were imported successfully :)


exploring the 3 datasets

In [6]:
print("Exploring the datasets needed for the competition..............")
# the datasets
train_data = pd.read_csv("train.csv") # for the training process
test_data = pd.read_csv("test.csv") # for the testing process
sample_submission = pd.read_csv("sample_submission.csv")
time.sleep(1) # delay for 1 second
# the dataset whose result will be needed for the submission process
print("The datasets were loaded succcessfully :)")

Exploring the datasets needed for the competition..............
The datasets were loaded succcessfully :)


Function to ease the encoding process

In [7]:
from sklearn import preprocessing, tree

def vformat_list(in_list):
  return_str =""
  columns = 3
  col_used = 0
  row_string = ""
  for item in in_list:
    row_string = row_string + f"{item:36}"
    col_used = col_used + 1
    if col_used == columns:
      return_str = return_str + row_string + "\n"
      col_used = 0
      row_string =""
  if (len(row_string)>0):
    return_str = return_str + row_string +"\n"
  return return_str


def printTree(clf, cols):
    text_representation = tree.export_text(clf, feature_names = cols, class_names = ['No Heart Disease', 'Heart Disease'], show_weights = True)
    print(text_representation)

def labelEncoder(df, colsList):
    le = preprocessing.LabelEncoder()
    for col in colsList:
        df[col] = le.fit_transform(df[col])

    return df

def oneHotEncoder(df, colsList):
    #Creates OneHotEncoder from sci-kit learn library
    one_hot_encoder = preprocessing.OneHotEncoder(sparse_output=False)

    #Applies OneHotEncoder to variable colsForOneHot
    encoded_data = one_hot_encoder.fit_transform(df[colsList])

    #Creates a new dataframe from the encoded data
    encoded_df = pd.DataFrame(
        encoded_data, columns=one_hot_encoder.get_feature_names_out(colsList))

    #Creates a new dataframe that combines the original dataframe with the encoded dataframe
    new_df = pd.concat([df.drop(colsList, axis=1), encoded_df], axis=1)

    return new_df

In [8]:
from sklearn import preprocessing, tree

def vformat_list(in_list):
  return_str =""
  columns = 3
  col_used = 0
  row_string = ""
  for item in in_list:
    row_string = row_string + f"{item:36}"
    col_used = col_used + 1
    if col_used == columns:
      return_str = return_str + row_string + "\n"
      col_used = 0
      row_string =""
  if (len(row_string)>0):
    return_str = return_str + row_string +"\n"
  return return_str


def printTree(clf, cols):
    text_representation = tree.export_text(clf, feature_names = cols, class_names = ['No Heart Disease', 'Heart Disease'], show_weights = True)
    print(text_representation)

def labelEncoder(df, colsList):
    le = preprocessing.LabelEncoder()
    for col in colsList:
        df[col] = le.fit_transform(df[col])

    return df

def oneHotEncoder(df, colsList):
    #Creates OneHotEncoder from sci-kit learn library
    one_hot_encoder = preprocessing.OneHotEncoder(sparse_output=False)

    #Applies OneHotEncoder to variable colsForOneHot
    encoded_data = one_hot_encoder.fit_transform(df[colsList])

    #Creates a new dataframe from the encoded data
    encoded_df = pd.DataFrame(
        encoded_data, columns=one_hot_encoder.get_feature_names_out(colsList))

    #Creates a new dataframe that combines the original dataframe with the encoded dataframe
    new_df = pd.concat([df.drop(colsList, axis=1), encoded_df], axis=1)

    return new_df

exploring the train dataset

In [9]:
print("Inspecting the training dataset...................")
print(f"The Table : \n {train_data}") # the training data
print()
print(f"The Shape : {train_data.shape}") # (nrows , ncols)
print()
print(f"The Info : \n {train_data.info()}") # the info of the dataset
print()
print(f"The Statistical Description : \n {train_data.describe()}") # the statistical description
print()
print(f"The Null Values in each column in the dataset : \n {train_data.isnull().sum()}") # the number of missing values in the dataset
print()

# the duplicated rows
duplicated_rows_train_dataset = train_data[train_data.duplicated()]
# if condition
if duplicated_rows_train_dataset.shape[0] == 0:
    print("There are no duplicated rows in the train dataset")
else:
  print(f"The number of duplicated values in the train dataset : {duplicated_rows_train_dataset.shape[0]}")
  print()
  print(f"The Duplicated Rows in the train dataset : \n {duplicated_rows_train_dataset}")

print()
print(f"The dtype of each column : \n {train_data.dtypes}")
print()
print(f"The columns in the dataset : \n {train_data.columns.to_list()}")
print()
time.sleep(1) # delay
print("The training dataset was inspected successfully :)")

Inspecting the training dataset...................
The Table : 
             id   age  daily_screen_time_hours  social_media_hours  \
0            0  24.0                      NaN                1.83   
1            1  19.0                     5.97                1.08   
2            2  18.0                     5.09                 NaN   
3            3  21.0                     6.42                1.26   
4            4  26.0                    11.20                1.87   
...        ...   ...                      ...                 ...   
691364  691364  22.0                    10.78                3.25   
691365  691365  20.0                     3.45                1.86   
691366  691366   NaN                      NaN                 NaN   
691367  691367  22.0                    11.78                3.69   
691368  691368  21.0                     5.64                1.03   

        gaming_hours  work_study_hours  sleep_hours  notifications_per_day  \
0               1.59        

In [10]:
# value counts check

# daily screen time hours
daily_hours_count = train_data["daily_screen_time_hours"].value_counts()
print(f"The value counts of the daily screen time hours : \n {daily_hours_count}")
print()
# social madia hours
social_hours_count = train_data["social_media_hours"].value_counts()
print(f"The value counts of the social media hours : \n {social_hours_count}")
print()

# gaming hours
gaming_hours_count = train_data["gaming_hours"].value_counts()
print(f"The value counts of the gaming hours : \n {gaming_hours_count}")
print()

# addicted label
addicted_label_count = train_data["addicted_label"].value_counts()
print(f"The value counts of the addicted label : \n {addicted_label_count}")
print()

# stress level
stress_level_count = train_data["stress_level"].value_counts()
print(f"The value counts of the stress level : \n {stress_level_count}")
print()

# weekend screen time
weekend_screen_time_count = train_data["weekend_screen_time"].value_counts()
print(f"The value counts of the weekend screen time : \n {weekend_screen_time_count}")
print()

# app opens per day
app_opens_per_day_count = train_data["app_opens_per_day"].value_counts()
print(f"The value counts of the app opens per day : \n {app_opens_per_day_count}")
print()

# academic work impact
academic_work_impact_count = train_data["academic_work_impact"].value_counts()
print(f"The value counts of the academic work impact : \n {academic_work_impact_count}")
print()


The value counts of the daily screen time hours : 
 daily_screen_time_hours
7.21     3434
9.30     3153
5.62     2978
7.76     2844
5.44     2616
         ... 
1.23        1
14.44       1
14.68       1
14.56       1
14.47       1
Name: count, Length: 1389, dtype: int64

The value counts of the social media hours : 
 social_media_hours
1.89    2872
1.68    2668
2.82    2615
2.19    2544
1.21    2411
        ... 
7.53       1
6.27       1
7.79       1
6.54       1
8.00       1
Name: count, Length: 721, dtype: int64

The value counts of the gaming hours : 
 gaming_hours
0.71    3358
1.07    3225
0.46    3116
0.47    3113
0.93    3054
        ... 
3.91      94
3.52      87
3.78      46
4.00       9
3.93       2
Name: count, Length: 401, dtype: int64

The value counts of the addicted label : 
 addicted_label
1    490474
0    200895
Name: count, dtype: int64

The value counts of the stress level : 
 stress_level
High      220873
Low       207783
Medium    207565
Name: count, dtype: int64

Th

# **cleaning the dataset**

1st : Fixing the inconsistent formatting

In [11]:
print("Found inconsistent formatting in the dataset")
print(f"The dtype of the columns before fixing the inconsistent formatting : \n {train_data.dtypes}")
print()
print("Fixing the inconsistent formatting...................")
# the fixing process
# Apply your explicit type requests
train_data["id"] = train_data["id"].astype(str)
train_data["age"] = train_data["age"].astype("Int64") # Use 'Int64' if age has missing values

# Map binary feature cleanly to 1/0 for Scikit-Learn compatibility
train_data["academic_work_impact"] = train_data["academic_work_impact"].map({"Yes": 1, "No": 0})
train_data["academic_work_impact"] = train_data["academic_work_impact"].astype("Int64")
# Set up optimized categories and nullable integers based on value counts
stress_order = ["Low", "Medium", "High"]
train_data["stress_level"] = pd.Categorical(train_data["stress_level"], categories=stress_order, ordered=True)
train_data["gender"] = train_data["gender"].astype("category")

train_data["notifications_per_day"] = train_data["notifications_per_day"].astype("Int64")
train_data["app_opens_per_day"] = train_data["app_opens_per_day"].astype("Int64")


# Continuous and discrete numeric columns needing scaling
numeric_features = [
    "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time"
]

# Nominal categorical columns (unordered text)
categorical_features = ["gender"]

# Ordinal categorical columns (ordered text)
ordinal_features = ["stress_level"]
stress_categories_order = [["Low", "Medium", "High"]]

# transforming through scikit-learn
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")), # Fills missing values safely
    ("scaler", StandardScaler())                  # Centers data around 0 with variance of 1
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")) # Avoids multi-collinearity
])

ordinal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(categories=stress_categories_order)) # Encodes Low=0, Med=1, High=2
])

# Combine all transformations into a single engine
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("ord", ordinal_transformer, ordinal_features),
    ],
    remainder="passthrough" # Safely leaves 'academic_work_impact' untouched as 1/0
)
time.sleep(1)
print("The inconsistent formatting was fixed successfully :)")
print(f"The dtype of the columns after fixing the inconsistent formatting : \n {train_data.dtypes}")
print()

Found inconsistent formatting in the dataset
The dtype of the columns before fixing the inconsistent formatting : 
 id                           int64
age                        float64
daily_screen_time_hours    float64
social_media_hours         float64
gaming_hours               float64
work_study_hours           float64
sleep_hours                float64
notifications_per_day      float64
app_opens_per_day          float64
weekend_screen_time        float64
gender                      object
stress_level                object
academic_work_impact        object
addicted_label               int64
dtype: object

Fixing the inconsistent formatting...................
The inconsistent formatting was fixed successfully :)
The dtype of the columns after fixing the inconsistent formatting : 
 id                           object
age                           Int64
daily_screen_time_hours     float64
social_media_hours          float64
gaming_hours                float64
work_study_hours     

2nd : Filling the missing values

In [12]:
# filling the missing values
print(f"The Null Values in each column in the dataset before filling the missing values : \n {train_data.isnull().sum()}")
print()
print("Fixing the missing values...................")

# the age
train_data["age"] = train_data["age"].fillna(train_data["age"].median())
# the gender
train_data["gender"] = train_data["gender"].fillna(train_data["gender"].mode()[0])
# daily screen time hours
train_data["daily_screen_time_hours"] = train_data["daily_screen_time_hours"].fillna(train_data["daily_screen_time_hours"].median())
# social media hours
train_data["social_media_hours"] = train_data["social_media_hours"].fillna(train_data["social_media_hours"].median())
# gaming hours
train_data["gaming_hours"] = train_data["gaming_hours"].fillna(train_data["gaming_hours"].median())
# sleep hours
train_data["sleep_hours"] = train_data["sleep_hours"].fillna(train_data["sleep_hours"].median())
# work study hours
train_data["work_study_hours"] = train_data["work_study_hours"].fillna(train_data["work_study_hours"].median())
# weekend screen time
train_data["weekend_screen_time"] = train_data["weekend_screen_time"].fillna(train_data["weekend_screen_time"].median())
# stress level
train_data["stress_level"] = train_data["stress_level"].fillna(train_data["stress_level"].mode()[0])
# academic work impact
train_data["academic_work_impact"] = train_data["academic_work_impact"].fillna(train_data["academic_work_impact"].mode()[0])
# app opens per day
train_data["app_opens_per_day"] = train_data["app_opens_per_day"].fillna(train_data["app_opens_per_day"].median())
# notifications per day
train_data["notifications_per_day"] = train_data["notifications_per_day"].fillna(train_data["notifications_per_day"].median())
time.sleep(1)
print("The missing values were filled successfully :)")
print(f"The Null Values in each column in the dataset after filling the missing values : \n {train_data.isnull().sum()}")
print()

The Null Values in each column in the dataset before filling the missing values : 
 id                              0
age                         28929
daily_screen_time_hours     95854
social_media_hours         133995
gaming_hours               126821
work_study_hours            51518
sleep_hours                 44480
notifications_per_day       67584
app_opens_per_day           80710
weekend_screen_time        112063
gender                      29034
stress_level                55148
academic_work_impact        44224
addicted_label                  0
dtype: int64

Fixing the missing values...................
The missing values were filled successfully :)
The Null Values in each column in the dataset after filling the missing values : 
 id                         0
age                        0
daily_screen_time_hours    0
social_media_hours         0
gaming_hours               0
work_study_hours           0
sleep_hours                0
notifications_per_day      0
app_opens_per_day  

# **downloading the cleaned dataset**

In [13]:
print("Downloading the cleaned training dataset...................")
train_data.to_csv("clean_training_data" , index=False)
time.sleep(1)
print("The cleaned training dataset was downloaded successfully :)")

The cleaned training dataset was downloaded successfully :)


# **Importing the cleaned training dataset**

In [14]:
df_train = pd.read_csv("clean_training_data")
print(f"The Table : \n {df_train}")
print()

The Table : 
             id  age  daily_screen_time_hours  social_media_hours  \
0            0   24                     7.77                1.83   
1            1   19                     5.97                1.08   
2            2   18                     5.09                2.31   
3            3   21                     6.42                1.26   
4            4   26                    11.20                1.87   
...        ...  ...                      ...                 ...   
691364  691364   22                    10.78                3.25   
691365  691365   20                     3.45                1.86   
691366  691366   27                     7.77                2.31   
691367  691367   22                    11.78                3.69   
691368  691368   21                     5.64                1.03   

        gaming_hours  work_study_hours  sleep_hours  notifications_per_day  \
0               1.59              2.11         7.46                    122   
1            

# **Small Inspection**

In [15]:
print(f"The shape of the cleaned training dataset : {df_train.shape}")
print()
print(f"The first 5 rows of the cleaned training dataset : \n {df_train.head()}")
print()
print(f"The last 5 rows of the cleaned training dataset : \n {df_train.tail()}")
print()
print(f"The info of the cleaned training dataset : \n {df_train.info()}")
print()
print(f"The statistical description of the cleaned training dataset : \n {df_train.describe()}")
print()
print(f"The columns in the cleaned training dataset : \n {df_train.columns.to_list()}")
print()
print(f"The dtype of each column in the cleaned training dataset : \n {df_train.dtypes}")
print()
print(f"The number of missing values in each column in the cleaned training dataset : \n {df_train.isnull().sum()}")
print()
print(f"The duplicated rows in the cleaned training dataset : \n {df_train[df_train.duplicated()]}")
print()


The shape of the cleaned training dataset : (691369, 14)

The first 5 rows of the cleaned training dataset : 
    id  age  daily_screen_time_hours  social_media_hours  gaming_hours  \
0   0   24                     7.77                1.83          1.59   
1   1   19                     5.97                1.08          1.33   
2   2   18                     5.09                2.31          1.33   
3   3   21                     6.42                1.26          1.42   
4   4   26                    11.20                1.87          2.81   

   work_study_hours  sleep_hours  notifications_per_day  app_opens_per_day  \
0              2.11         7.46                    122                 38   
1              3.03         8.22                     76                 19   
2              2.20         6.25                    134                 60   
3              3.36         8.85                    112                 94   
4              1.95         5.25                    150     

In [16]:
gender_values = df_train["gender"].value_counts()
print(f"The value counts of the gender column : \n {gender_values}")

The value counts of the gender column : 
 gender
Male      252696
Female    221595
Other     217078
Name: count, dtype: int64


# **Preparing the dataset for the modeling process**

Feature Scaling using StandardScaler

In [17]:
print("Applying the ColumnTransformer for combined scaling and encoding...")

# Separate features (X) and target (y) from df_train
# 'id' should be kept separate if needed later, but not part of features for model training
X = df_train.drop(columns=['id', 'addicted_label'])
y = df_train['addicted_label']

# Apply the preprocessor to the features
# The preprocessor handles scaling for numeric and encoding for categorical/ordinal features
X_preprocessed = preprocessor.fit_transform(X)

# Get the feature names after preprocessing
# ColumnTransformer's get_feature_names_out() method is ideal for this
feature_names_out = preprocessor.get_feature_names_out()

# Convert the preprocessed array back to a DataFrame
df_processed = pd.DataFrame(X_preprocessed, columns=feature_names_out, index=X.index)

# Display the head of the preprocessed DataFrame
print("\nPreprocessed Feature DataFrame Head:")
print(df_processed.head())

# Display the data types to verify transformations
print("\nPreprocessed Feature DataFrame dtypes:")
print(df_processed.dtypes)

# Now df_processed contains all the features scaled and encoded as per the ColumnTransformer.
# 'y' contains the target variable.
# 'id' from df_train can be rejoined if necessary.

Applying the ColumnTransformer for combined scaling and encoding...

Preprocessed Feature DataFrame Head:
   num__age  num__daily_screen_time_hours  num__social_media_hours  \
0 -0.521628                      0.044032                -0.515298   
1 -1.512751                     -0.668516                -1.149041   
2 -1.710976                     -1.016872                -0.109703   
3 -1.116302                     -0.490379                -0.996942   
4 -0.125179                      1.401831                -0.481498   

   num__gaming_hours  num__work_study_hours  num__sleep_hours  \
0           0.182566              -0.201793          0.549303   
1          -0.124770               0.557420          1.185745   
2          -0.124770              -0.127522         -0.463979   
3          -0.018385               0.829746          1.713321   
4           1.624684              -0.333830         -1.301402   

   num__notifications_per_day  num__app_opens_per_day  \
0                   -0.38

### **Inspect the shape and columns of the processed DataFrame**

In [18]:
print(f"Shape of df_processed: {df_processed.shape}")
print("\nColumns in df_processed:")
print(df_processed.columns.tolist())

Shape of df_processed: (691369, 13)

Columns in df_processed:
['num__age', 'num__daily_screen_time_hours', 'num__social_media_hours', 'num__gaming_hours', 'num__work_study_hours', 'num__sleep_hours', 'num__notifications_per_day', 'num__app_opens_per_day', 'num__weekend_screen_time', 'cat__gender_Male', 'cat__gender_Other', 'ord__stress_level', 'remainder__academic_work_impact']


## Data Splitting

We will split the processed data into training and testing sets to evaluate the model's performance on unseen data. A common split ratio is 80% for training and 20% for testing.

In [19]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df_processed, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (553095, 13)
Shape of X_test: (138274, 13)
Shape of y_train: (553095,)
Shape of y_test: (138274,)


## Model Training (MLPClassifier)

We will train a Multi-layer Perceptron (MLP) Classifier, which is a type of artificial neural network, on our preprocessed training data. We'll start with a basic configuration.

In [20]:
# Initialize and train the MLPClassifier
# Using a simple architecture for demonstration
mlp_model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=100, random_state=42, verbose=True)

print("Training the MLPClassifier...")
start_time = time.time()
mlp_model.fit(X_train, y_train)
end_time = time.time()
print(f"MLPClassifier training completed in {end_time - start_time:.2f} seconds.")

Training the MLPClassifier...
Iteration 1, loss = 0.31334993
Iteration 2, loss = 0.29928507
Iteration 3, loss = 0.29516309
Iteration 4, loss = 0.29331517
Iteration 5, loss = 0.29211803
Iteration 6, loss = 0.29131359
Iteration 7, loss = 0.29060276
Iteration 8, loss = 0.29012278
Iteration 9, loss = 0.28957085
Iteration 10, loss = 0.28908590
Iteration 11, loss = 0.28854940
Iteration 12, loss = 0.28815571
Iteration 13, loss = 0.28767350
Iteration 14, loss = 0.28736053
Iteration 15, loss = 0.28717943
Iteration 16, loss = 0.28677243
Iteration 17, loss = 0.28646860
Iteration 18, loss = 0.28628279
Iteration 19, loss = 0.28609037
Iteration 20, loss = 0.28586769
Iteration 21, loss = 0.28576053
Iteration 22, loss = 0.28565019
Iteration 23, loss = 0.28538643
Iteration 24, loss = 0.28523043
Iteration 25, loss = 0.28510745
Iteration 26, loss = 0.28495377
Iteration 27, loss = 0.28490825
Iteration 28, loss = 0.28474253
Iteration 29, loss = 0.28462272
Iteration 30, loss = 0.28447187
Iteration 31, loss 

## Model Evaluation

Now, let's evaluate the performance of our trained MLPClassifier on the testing set. We'll use a classification report and a confusion matrix to understand its accuracy, precision, recall, and F1-score.

In [21]:
# Make predictions on the test set
y_pred = mlp_model.predict(X_test)

# Evaluate the model
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred), index=['Actual 0', 'Actual 1'], columns=['Predicted 0', 'Predicted 1']))


Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.73      0.76     40289
           1       0.89      0.92      0.90     97985

    accuracy                           0.86    138274
   macro avg       0.84      0.82      0.83    138274
weighted avg       0.86      0.86      0.86    138274


Confusion Matrix:


,Predicted 0,Predicted 1
Actual 0,29516,10773
Actual 1,8119,89866


## Training and Evaluation of an Alternative Model: RandomForestClassifier

Let's try a different classification algorithm, the RandomForestClassifier, to see if we can achieve better performance or a more balanced classification report.

In [23]:
# Import RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier

# Initialize and train the RandomForestClassifier
# Using a reasonable number of estimators for a first pass
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced') # n_jobs=-1 uses all available cores and class_weight='balanced' to handle imbalance

print("Training the RandomForestClassifier...")
start_time = time.time()
rf_model.fit(X_train, y_train)
end_time = time.time()
print(f"RandomForestClassifier training completed in {end_time - start_time:.2f} seconds.")

Training the RandomForestClassifier...
RandomForestClassifier training completed in 97.36 seconds.


### Evaluate RandomForestClassifier Performance

Now, let's evaluate the performance of our trained RandomForestClassifier on the testing set and compare it with the MLPClassifier.

In [ ]:
# Make predictions on the test set using RandomForestClassifier
y_pred_rf = rf_model.predict(X_test)

# Evaluate the RandomForestClassifier model
print("\nRandomForestClassifier - Classification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nRandomForestClassifier - Confusion Matrix:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_rf), index=['Actual 0', 'Actual 1'], columns=['Predicted 0', 'Predicted 1']))

Label Encoding and One Hot Encoding

In [ ]:
print("Re-applying necessary dtype conversions for the preprocessor...")

# Re-apply categorical dtypes as per original preprocessor setup in XBlF4z0aWaOb
# This ensures gender and stress_level are correctly typed for ColumnTransformer
stress_order = ["Low", "Medium", "High"]
df_train["stress_level"] = pd.Categorical(df_train["stress_level"], categories=stress_order, ordered=True)
df_train["gender"] = df_train["gender"].astype("category")

print(f"df_train dtypes after re-applying type conversions:\n{df_train.dtypes}")